# Exercises XP : Evaluating LLMs for Summarization

**Solution complète** — évaluation de LLM sur des tâches de résumé (accuracy vs ROUGE).


## Ce que vous allez apprendre
- Évaluation pratique de la synthèse : accuracy vs ROUGE.
- Forces / faiblesses des métriques et comparaison de tailles de modèles.
- Utilisation de Hugging Face `transformers` + `evaluate`.
- Chargement, échantillonnage, prétraitement des données, et débogage des sorties.


## Partie I — Configuration
Installation des bibliothèques et téléchargement des ressources NLTK.


In [ ]:
# Partie I. Setup (à exécuter une fois par runtime)
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

import warnings
warnings.filterwarnings('ignore')


## Partie II — Chargement et exploration du jeu de données

Dataset : [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) version `3.0.0`
(mapping `article` -> `prompt_text`, `highlights` -> `prompt_title`).

> **Note sur les versions** : `1.0.0` / `2.0.0` / `3.0.0` diffèrent surtout par le nettoyage
> des données et la présence de l'`id`. On utilise **3.0.0**, la version standard des benchmarks.

- 100 échantillons d'entraînement, 50 de test (pour alléger le calcul).
- Un CSV local `train.csv` / `test.csv` peut être utilisé à la place.


In [ ]:
import pandas as pd
from datasets import load_dataset

pd.set_option('display.max_colwidth', 120)

# Laisser vide pour utiliser Hugging Face
train_path = ''   # ex: '/content/train.csv'
test_path  = ''   # ex: '/content/test.csv'

fallback = pd.DataFrame([
    {'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
     'prompt_title': 'Cat rests on mat at sunset'},
    {'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
     'prompt_title': 'Water found on the moon'},
    {'prompt_text': 'The local team won the championship after a dramatic final match.',
     'prompt_title': 'Local team clinches title'},
])

def load_and_sample(path, split_name, n, seed=42):
    """Charge un CSV local ou un slice de cnn_dailymail, puis échantillonne n lignes."""
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = (ds.to_pandas()[['article', 'highlights']]
                    .rename(columns={'article': 'prompt_text',
                                     'highlights': 'prompt_title'}))
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=seed).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df  = load_and_sample(test_path,  'test',  50)

print("train:", train_df.shape, "| test:", test_df.shape)


In [ ]:
# Exploration : premier exemple de l'échantillon d'entraînement
row = train_df.iloc[0]

print("=" * 80)
print("ARTICLE (prompt_text)")
print("=" * 80)
print(row['prompt_text'][:1200], "...\n")

print("=" * 80)
print("RÉSUMÉ DE RÉFÉRENCE (prompt_title)")
print("=" * 80)
print(row['prompt_title'])

print("\nLongueur article (mots) :", len(row['prompt_text'].split()))
print("Longueur résumé  (mots) :", len(row['prompt_title'].split()))


In [ ]:
# Inspection de la structure
display(train_df.head(3))
display(test_df.head(3))

train_df['n_words_text']  = train_df['prompt_text'].str.split().str.len()
train_df['n_words_title'] = train_df['prompt_title'].str.split().str.len()
display(train_df[['n_words_text', 'n_words_title']].describe())


## Partie III — Résumé avec T5

- `batch_generator` : découpe une liste en mini-batches.
- `summarize_with_t5` : charge le tokenizer + modèle, gère le GPU (CUDA), préfixe
  les entrées par `"summarize: "`, génère, décode avec `skip_special_tokens=True`.
- Nettoyage mémoire (`torch.cuda.empty_cache()` + `gc.collect()`) après chaque batch.


In [ ]:
import torch, gc
from typing import Iterator, List
from transformers import AutoTokenizer, T5ForConditionalGeneration

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", DEVICE)


def batch_generator(items: List[str], batch_size: int) -> Iterator[List[str]]:
    """Yield des tranches successives de `items` de taille `batch_size`."""
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def _cleanup():
    """Libère le cache CUDA + garbage collection."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def summarize_with_t5(texts: List[str],
                      model_name: str = 't5-small',
                      batch_size: int = 4,
                      max_new_tokens: int = 32,
                      max_input_length: int = 512) -> List[str]:
    """Génère des résumés avec un modèle T5 (seq2seq)."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(DEVICE)
    model.eval()

    summaries: List[str] = []

    for batch in batch_generator(texts, batch_size):
        # T5 est multi-tâches : le préfixe indique la tâche demandée
        prompts = ["summarize: " + t for t in batch]

        enc = tokenizer(prompts,
                        max_length=max_input_length,
                        truncation=True,
                        padding=True,
                        return_tensors='pt').to(DEVICE)

        with torch.no_grad():
            out_ids = model.generate(**enc,
                                     max_new_tokens=max_new_tokens,
                                     num_beams=2,
                                     early_stopping=True)

        summaries.extend(tokenizer.batch_decode(out_ids, skip_special_tokens=True))

        del enc, out_ids
        _cleanup()

    del model, tokenizer
    _cleanup()
    return summaries


In [ ]:
# Génération des résumés avec t5-small
RUN_T5 = True

if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(),
                                           model_name='t5-small',
                                           batch_size=4)

    t5_results = pd.DataFrame({
        'prompt_text': train_df['prompt_text'].str.slice(0, 100) + '...',
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5,
    })
    display(t5_results.head(10))
else:
    print("Skipping T5 generation. Set RUN_T5=True to execute.")


## Partie IV — Évaluation de la précision (accuracy)

Accuracy naïve = correspondance **exacte** entre chaîne générée et chaîne de référence.


In [ ]:
def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    """Exact-match accuracy : proportion de prédictions strictement identiques."""
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)


if 'train_summaries_t5' in globals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy (t5-small) : {acc:.4f}")

    # Variante un peu plus tolérante : insensible à la casse et à la ponctuation
    import re
    def norm(s):
        return re.sub(r'[^a-z0-9 ]', '', s.lower()).strip()
    acc_soft = compute_accuracy([norm(p) for p in train_summaries_t5],
                                [norm(r) for r in train_df['prompt_title']])
    print(f"Accuracy normalisée (casse/ponctuation) : {acc_soft:.4f}")
else:
    print("Accuracy skipped (no predictions).")


### Interprétation

L'accuracy est **~0.0000**, et ce n'est pas un bug.

1. **Espace de sortie quasi infini.** Un résumé est du texte libre : il existe des
   milliards de formulations valides pour une même idée. Exiger une égalité de chaînes
   revient à n'accepter qu'une seule de ces formulations.
2. **Métrique tout-ou-rien.** Un résumé qui capture parfaitement le sens mais change
   un seul mot (« a déclaré » vs « a dit ») reçoit exactement le même score — zéro —
   qu'un résumé totalement hors-sujet. La métrique ne discrimine pas entre bon et mauvais.
3. **Aucun gradient d'information.** On ne peut pas comparer deux modèles avec une
   métrique qui renvoie 0 pour les deux. Elle est inutilisable pour le classement.
4. **Ponctuation, casse, longueur** ajoutent du bruit sans rapport avec la qualité sémantique.

L'accuracy convient à la **classification** (classes discrètes, une seule réponse correcte).
Pour la génération de texte, il faut une métrique de **chevauchement partiel** → ROUGE.


## Partie V — Mise en œuvre de la métrique ROUGE

**ROUGE** = *Recall-Oriented Understudy for Gisting Evaluation*. Mesure le
chevauchement de n-grammes entre prédiction et référence.

| Variante | Ce qu'elle mesure |
|---|---|
| ROUGE-1 | chevauchement d'unigrammes (contenu) |
| ROUGE-2 | chevauchement de bigrammes (fluidité / ordre local) |
| ROUGE-L | plus longue sous-séquence commune (structure) |
| ROUGE-Lsum | ROUGE-L calculé **phrase par phrase** |

**Prétraitement** : `rouge_score` attend les phrases séparées par des `\n` pour que
`rougeLsum` identifie correctement les frontières de phrases. On utilise `nltk.sent_tokenize`.

> Attention : le notebook de départ contenait `return "".join(sents)` — un bug.
> Il faut `"\n".join(sents)`.


In [ ]:
import evaluate
from nltk.tokenize import sent_tokenize

rouge = evaluate.load('rouge')


def normalize_text(text: str) -> str:
    """Sépare les phrases par des retours à la ligne (requis par rougeLsum)."""
    if not isinstance(text, str) or not text.strip():
        return ""
    sents = sent_tokenize(text.strip())
    return "\n".join(sents)          # <-- "\n", pas "" !


def compute_rouge_score(preds: List[str],
                        refs: List[str],
                        use_stemmer: bool = True) -> dict:
    """Calcule les scores ROUGE agrégés après normalisation."""
    preds_n = [normalize_text(p) for p in preds]
    refs_n  = [normalize_text(r) for r in refs]
    return rouge.compute(predictions=preds_n,
                         references=refs_n,
                         use_stemmer=use_stemmer)


# Smoke test
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check:")
for k, v in compute_rouge_score(test_preds, test_refs).items():
    print(f"  {k:12s}: {v:.4f}")


## Partie VI — Comprendre les scores ROUGE

Quatre expériences : correspondance exacte, prédiction vide, effet du *stemming*,
comportement des n-grammes, et symétrie.


In [ ]:
def show(title, scores):
    print(f"\n--- {title} ---")
    for k, v in scores.items():
        print(f"  {k:12s}: {v:.4f}")

# 1) Correspondance exacte
exact = ["The cat sat on the mat.", "Water was found on the moon."]
show("1. Correspondance EXACTE (pred == ref)", compute_rouge_score(exact, exact))

# 2) Prédiction vide
show("2. Prédiction VIDE", compute_rouge_score(["", ""], exact))


**Bornes de la métrique.** Correspondance exacte → **1.0** partout (le maximum).
Prédiction vide → **0.0** partout (le minimum). ROUGE est donc bien borné sur [0, 1],
et tout résultat intermédiaire mesure un chevauchement partiel.


In [ ]:
# 3) Effet de la racinisation (stemming)
pred_stem = ["The runners are running quickly"]
ref_stem  = ["The runner runs quick"]

show("3a. SANS stemmer", compute_rouge_score(pred_stem, ref_stem, use_stemmer=False))
show("3b. AVEC stemmer", compute_rouge_score(pred_stem, ref_stem, use_stemmer=True))


**Effet du stemming.** Sans racinisation, `running` ≠ `runs` et `runners` ≠ `runner` :
seuls les mots identiques comptent, le score s'effondre. Avec le stemmer (Porter),
`run/runs/running/runners → run` et `quick/quickly → quick`, donc les correspondances
morphologiques sont récupérées et ROUGE-1 grimpe fortement.

Le stemming rend ROUGE **plus tolérant aux variations grammaticales** et donc plus proche
d'un jugement humain — au prix d'une perte de précision (il confond parfois des mots de
sens différents partageant une racine).


In [ ]:
# 4) Analyse N-gram : chevauchement progressif
reference = "The quick brown fox jumps over the lazy dog"

variants = {
    "identique              ": "The quick brown fox jumps over the lazy dog",
    "mots réordonnés        ": "dog lazy the over jumps fox brown quick The",
    "moitié des mots (ordre)": "The quick brown fox jumps",
    "mêmes mots, autres liens": "The fox is quick and brown, the dog is lazy",
    "synonymes (0 recouvr.) ": "A fast auburn canine leaps above a sleepy hound",
    "aucun rapport          ": "Stock markets closed lower on Tuesday",
}

rows = []
for name, pred in variants.items():
    s = compute_rouge_score([pred], [reference])
    rows.append({'variante': name.strip(),
                 'rouge1': round(s['rouge1'], 3),
                 'rouge2': round(s['rouge2'], 3),
                 'rougeL': round(s['rougeL'], 3)})

display(pd.DataFrame(rows))


**Lecture des résultats (ROUGE-1 vs ROUGE-2).**

- **Mots réordonnés** : ROUGE-1 reste **très élevé** (tous les unigrammes sont présents)
  mais ROUGE-2 **tombe à ~0** — aucune paire de mots consécutifs ne survit à la permutation.
  C'est la démonstration clé : *ROUGE-1 mesure le contenu, ROUGE-2 mesure l'ordre.*
- **Moitié des mots dans l'ordre** : ROUGE-1 est réduit de moitié, mais ROUGE-2 reste
  correct car les bigrammes conservés sont intacts.
- **Synonymes** : ROUGE ≈ 0 alors que le sens est identique. **Faiblesse majeure** :
  ROUGE est purement lexical, il ignore la sémantique.
- ROUGE-2 est **toujours ≤ ROUGE-1** : il y a strictement moins de bigrammes que
  d'unigrammes à faire correspondre, et une correspondance de bigramme est une contrainte
  plus forte.


In [ ]:
# 5) Symétrie
A = ["The cat sat on the mat quietly"]
B = ["The cat sat on the mat"]

s_ab = compute_rouge_score(A, B)   # A prédit, B référence
s_ba = compute_rouge_score(B, A)   # B prédit, A référence

sym = pd.DataFrame({'pred=A, ref=B': s_ab, 'pred=B, ref=A': s_ba}).round(4)
sym['identique ?'] = (sym.iloc[:, 0] == sym.iloc[:, 1])
display(sym)


**Symétrie.** Les scores sont **identiques** dans les deux sens.

Raison : la bibliothèque `evaluate`/`rouge_score` renvoie par défaut le **F-mesure**
(moyenne harmonique de la précision et du rappel). Or, échanger prédiction et référence
échange précision et rappel — et la moyenne harmonique est symétrique en ses deux arguments.

Conséquence pratique : le score F **ne dit pas** si un modèle est verbeux (bon rappel,
faible précision) ou trop laconique (bonne précision, faible rappel). Pour diagnostiquer ce
comportement, il faut inspecter précision et rappel séparément (`use_aggregator=False` +
`rouge_score` en direct).


## Partie VII — Comparaison petits vs grands modèles

Modèles : `t5-small` (60M), `t5-base` (220M), `gpt2` (124M, décodeur seul).

**Spécificités de GPT-2** :
- Ce n'est pas un modèle seq2seq → on utilise le prompt `"...\nTL;DR:"`.
- Pas de token de padding par défaut → `tokenizer.pad_token = tokenizer.eos_token`.
- Padding **à gauche** obligatoire pour la génération par batch.
- Fenêtre de 1024 tokens : on tronque l'article pour laisser la place à la génération.
- La sortie contient le prompt → il faut **découper** pour ne garder que la continuation.


In [ ]:
from transformers import AutoModelForCausalLM

def summarize_with_gpt2(texts: List[str],
                        model_name: str = 'gpt2',
                        batch_size: int = 2,
                        max_new_tokens: int = 32) -> List[str]:
    """Résumé « TL;DR » avec un modèle causal (GPT-2)."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'          # indispensable pour generate() en batch

    model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
    model.config.pad_token_id = tokenizer.eos_token_id
    model.eval()

    # Garde-fou : fenêtre de contexte 1024 tokens
    ctx = model.config.n_positions            # 1024
    max_input_length = ctx - max_new_tokens - 8

    summaries: List[str] = []

    for batch in batch_generator(texts, batch_size):
        prompts = []
        for t in batch:
            # tronque l'article AVANT d'ajouter le suffixe TL;DR
            ids = tokenizer.encode(t, truncation=True, max_length=max_input_length - 10)
            prompts.append(tokenizer.decode(ids) + "\nTL;DR:")

        enc = tokenizer(prompts,
                        return_tensors='pt',
                        padding=True,
                        truncation=True,
                        max_length=max_input_length).to(DEVICE)

        with torch.no_grad():
            out_ids = model.generate(**enc,
                                     max_new_tokens=max_new_tokens,
                                     do_sample=False,
                                     pad_token_id=tokenizer.eos_token_id)

        # on ne garde que les tokens générés APRÈS le prompt
        gen_only = out_ids[:, enc['input_ids'].shape[1]:]
        decoded = tokenizer.batch_decode(gen_only, skip_special_tokens=True)
        summaries.extend([d.strip().split('\n')[0].strip() for d in decoded])

        del enc, out_ids, gen_only
        _cleanup()

    del model, tokenizer
    _cleanup()
    return summaries


In [ ]:
def compute_rouge_per_row(df: pd.DataFrame,
                          pred_col: str,
                          ref_col: str = 'prompt_title') -> pd.DataFrame:
    """Ajoute des colonnes ROUGE-1/2/L calculées ligne par ligne."""
    out = df.copy()
    r1, r2, rl, rls = [], [], [], []

    for pred, ref in zip(out[pred_col], out[ref_col]):
        s = compute_rouge_score([pred], [ref])
        r1.append(s['rouge1'])
        r2.append(s['rouge2'])
        rl.append(s['rougeL'])
        rls.append(s['rougeLsum'])

    out[f'{pred_col}_rouge1']    = r1
    out[f'{pred_col}_rouge2']    = r2
    out[f'{pred_col}_rougeL']    = rl
    out[f'{pred_col}_rougeLsum'] = rls
    return out


In [ ]:
RUN_COMPARE = True

# On travaille sur un sous-échantillon pour la vitesse
N = 20
sub = train_df.head(N).copy()

if RUN_COMPARE:
    texts = sub['prompt_text'].tolist()

    print("→ t5-small ...")
    sub['t5_small'] = summarize_with_t5(texts, 't5-small', batch_size=4)

    print("→ t5-base ...")
    sub['t5_base']  = summarize_with_t5(texts, 't5-base',  batch_size=2)

    print("→ gpt2 ...")
    sub['gpt2']     = summarize_with_gpt2(texts, 'gpt2', batch_size=2)

    print("Terminé.")
    display(sub[['prompt_title', 't5_small', 't5_base', 'gpt2']].head())


In [ ]:
# Scores ROUGE agrégés par modèle
MODELS = ['t5_small', 't5_base', 'gpt2']

rouge_dict = {
    m: compute_rouge_score(sub[m].tolist(), sub['prompt_title'].tolist())
    for m in MODELS
}

for m, s in rouge_dict.items():
    print(f"\n{m}:")
    for k, v in s.items():
        print(f"  {k:12s}: {v:.4f}")


In [ ]:
# ROUGE par ligne
scored = sub.copy()
for m in MODELS:
    scored = compute_rouge_per_row(scored, m)

cols = ['prompt_title'] + [f'{m}_rouge{n}' for m in MODELS for n in ['1', '2', 'L']]
display(scored[cols].round(3).head(10))


## Partie VIII — Comparaison de tous les modèles


In [ ]:
def compare_models(rouge_dict: dict) -> pd.DataFrame:
    """{model_name: rouge_scores_dict} -> DataFrame des scores moyens."""
    return (pd.DataFrame(rouge_dict).T
              .rename_axis('model')
              .reset_index()
              .round(4)
              .sort_values('rouge1', ascending=False))


def compare_models_summaries(df: pd.DataFrame, pred_cols: list) -> pd.DataFrame:
    """Vue côte à côte : référence + résumé de chaque modèle."""
    cols = ['prompt_title'] + list(pred_cols)
    return df[cols].rename(columns={'prompt_title': 'reference'})


In [ ]:
print("=== Scores ROUGE moyens par modèle ===")
comparison = compare_models(rouge_dict)
display(comparison)

print("\n=== Résumés côte à côte ===")
display(compare_models_summaries(sub, MODELS).head(8))


In [ ]:
# Visualisation
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
comparison.set_index('model')[['rouge1', 'rouge2', 'rougeL']].plot(kind='bar', ax=ax)
ax.set_ylabel('Score ROUGE (F1)')
ax.set_title('Comparaison des modèles sur la tâche de synthèse')
ax.set_xlabel('')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Quel modèle gagne, et pourquoi ?

**Classement attendu : `t5-base` > `t5-small` >> `gpt2`.**

1. **t5-base bat t5-small** (~220M vs ~60M paramètres). Plus de capacité → meilleure
   abstraction, sélection plus fine des informations saillantes. Le gain est réel mais
   **modéré** : les rendements sont décroissants, et l'écart en ROUGE-1 est souvent de
   quelques points seulement pour ~4× le coût de calcul.

2. **GPT-2 est loin derrière.** Trois raisons structurelles, pas une question de taille :
   - **Pas de fine-tuning sur la synthèse.** T5 a vu la tâche `summarize:` pendant son
     pré-entraînement multi-tâches. GPT-2 est un modèle de langue brut : le prompt `TL;DR:`
     est un bricolage *zero-shot*.
   - **Architecture décodeur seul.** Pas d'encodeur bidirectionnel pour construire une
     représentation globale de l'article avant de générer.
   - **Fenêtre de 1024 tokens.** Les articles CNN/DailyMail sont souvent tronqués, donc
     l'information à résumer est parfois absente du contexte.
   
   GPT-2 a tendance à **continuer** l'article plutôt qu'à le résumer, ou à produire du
   texte dégénéré. Cela illustre que **la tâche compte plus que la taille** : un petit
   modèle spécialisé écrase un modèle généraliste plus grand mais non aligné.

3. **Attention au biais de ROUGE.** ROUGE récompense la copie littérale. Les modèles
   extractifs obtiennent mécaniquement de bons scores. Un score élevé ≠ meilleur résumé
   pour un humain.


## Conclusion / Réflexion

**Quelles métriques ont été les plus informatives ?**
ROUGE, sans hésitation. L'accuracy exacte renvoie 0 pour tous les modèles : elle ne
permet aucun classement. ROUGE fournit un signal continu et gradué. Dans la famille ROUGE,
**ROUGE-1 et ROUGE-2 sont complémentaires** : le premier mesure la couverture du contenu,
le second la fluidité et l'ordre. ROUGE-L capture la structure via la plus longue
sous-séquence commune. Regarder les trois évite de se faire piéger par un modèle qui
récite les bons mots dans le désordre.

**Impact de la taille du modèle.**
Positif mais avec des rendements décroissants (`t5-base` > `t5-small`). Le facteur
dominant n'est pas la taille brute mais **l'alignement architecture/tâche** : `gpt2`,
plus gros que `t5-small`, obtient des scores nettement inférieurs parce qu'il n'a jamais
appris à résumer. La qualité qualitative suit globalement ROUGE, mais avec des exceptions
notables — j'ai vu des résumés fluides et corrects sémantiquement pénalisés pour avoir
utilisé des synonymes.

**Où l'accuracy s'effondre.**
Dès la première ligne. Elle suppose une réponse unique et correcte, alors qu'un résumé
admet une infinité de formulations valides. C'est une métrique de classification appliquée
à un problème de génération. Elle est également **non graduée** : un résumé parfait à un
mot près et un résumé absurde reçoivent le même 0.

**Limites de ROUGE.**
ROUGE reste purement **lexical**. Il ne voit pas les synonymes (« automobile » vs « voiture »
scorent 0), ne détecte pas les **hallucinations factuelles** (un résumé qui copie les bons
mots mais inverse une négation obtient un excellent score), et ne juge ni la cohérence
ni la lisibilité.

**Extensions possibles.**
- **Métriques sémantiques** : BERTScore, BLEURT, MoverScore — embeddings contextuels
  plutôt que chevauchement de surface.
- **LLM-as-a-judge** : demander à un modèle fort de noter fidélité, concision, fluidité
  sur une grille. Corrèle mieux avec l'humain, mais introduit ses propres biais.
- **Vérification factuelle** : QAGS, FactCC, ou entailment (NLI) entre article et résumé
  pour détecter les hallucinations.
- **Évaluation humaine** : annotations sur 3–4 axes (pertinence, cohérence, fluidité,
  fidélité), avec mesure de l'accord inter-annotateurs (Krippendorff's α).
- **Sondes adverses** : injecter des négations, permuter des entités nommées, ou inverser
  des dates dans l'article et vérifier que le score du résumé chute. Si ROUGE ne bouge pas,
  c'est la preuve qu'il ne mesure pas la fidélité factuelle.
